In [36]:
import pandas as pd
import numpy as np
import openpyxl
import seaborn as sns
import matplotlib.pyplot as plt

In [52]:
# read the dataset
df = pd.read_csv('kick.csv')
# show all columns information
df.info

C:\Users\49157\AppData\Local\Temp\ipykernel_10852\1508352892.py:2: DtypeWarning: Columns (27) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('kick.csv')


<bound method DataFrame.info of        PurchaseID  PurchaseTimestamp      PurchaseDate Auction  VehYear  \
0               0         1253232000  18/09/2009 10:00   OTHER   2008.0   
1               1         1253232000  18/09/2009 10:00   OTHER   2008.0   
2               2         1253232000  18/09/2009 10:00   OTHER   2008.0   
3               3         1253232000  18/09/2009 10:00   OTHER   2008.0   
4               4         1253232000  18/09/2009 10:00   OTHER   2008.0   
...           ...                ...               ...     ...      ...   
41471       41471         1259712000   2/12/2009 10:00   ADESA   2001.0   
41472       41472         1259712000   2/12/2009 10:00   ADESA   2007.0   
41473       41473         1259712000   2/12/2009 10:00   ADESA   2005.0   
41474       41474         1259712000   2/12/2009 10:00   ADESA   2006.0   
41475       41475         1259712000   2/12/2009 10:00   ADESA   2006.0   

            Make   Color Transmission WheelTypeID WheelType  ...  \

In [53]:
# describe key statistics from IsBadBuy column
print(df['IsBadBuy'].describe())

count    41476.000000
mean         0.129497
std          0.335753
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000
Name: IsBadBuy, dtype: float64


In [54]:
print(df['IsBadBuy'].value_counts())

IsBadBuy
0    36105
1     5371
Name: count, dtype: int64


In [55]:
# Drop irrelevant columns
df.drop(['PurchaseID', 'PurchaseTimestamp', 'WheelTypeID', 'TopThreeAmericanName', 'Nationality', 'ForSale', 'MMRCurrentRetailRatio'], axis=1, inplace=True, errors='ignore')

In [56]:
# Divide column PuchaseDate into month and year, then drop the original column
df['PurchaseMonth'] = pd.to_datetime(df['PurchaseDate'], dayfirst=True).dt.month
df['PurchaseYear'] = pd.to_datetime(df['PurchaseDate'], dayfirst=True).dt.year
df.drop(['PurchaseDate'], axis=1, inplace=True, errors='ignore')

In [57]:
# Rename column 'MMRAquisitonRetailCleanPrice' to 'MMRAcquisitionRetailCleanPrice'
df.rename(columns={'MMRAcquisitonRetailCleanPrice': 'MMRAcquisitionRetailCleanPrice'}, inplace=True)

In [58]:
# Drop columns with the risk of data leakage
#df.drop(['WarrantyCost','MMRCurrentAuctionAveragePrice', 'MMRCurrentAuctionCleanPrice', 'MMRCurrentRetailAveragePrice', 'MMRCurrentRetailCleanPrice', 'MMRCurrentRetailRatio'], axis=1, inplace=True, errors='ignore')

Correcting the values

In [59]:
# In any column replace ? with NaN
df.replace('?', np.nan, inplace=True)

In [60]:
categorical_columns = ['PurchaseMonth', 'PurchaseYear', 'Auction', 'VehYear', 'Make', 'Color', 'Transmission', 'WheelType', 'Size', 'PRIMEUNIT', 'AUCGUART', 'VNST', 'IsOnlineSale']

In [61]:
# For categorical columns: replace Nan with the mode of the column
for col in categorical_columns: 
    df[col] = df[col].fillna(df[col].mode()[0])
    df[col] = df[col].astype('category')

In [62]:
numerical_columns = ['VehOdo', 'MMRAcquisitionAuctionAveragePrice', 'MMRAcquisitionAuctionCleanPrice', 'MMRAcquisitionRetailAveragePrice', 'MMRAcquisitionRetailCleanPrice', 'MMRCurrentAuctionAveragePrice','MMRCurrentAuctionCleanPrice','MMRCurrentRetailAveragePrice','MMRCurrentRetailCleanPrice', 'VehBCost', 'WarrantyCost']

In [63]:
# For numerical columns: replace Nan with the mean of the column
for col in numerical_columns: 
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col] = df[col].fillna(df[col].mean())
    df[col] = df[col].astype('float64')

In [64]:
# Delete outliers in numerical columns using IQR method
'''for col in numerical_columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df = df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]'''

'for col in numerical_columns:\n    Q1 = df[col].quantile(0.25)\n    Q3 = df[col].quantile(0.75)\n    IQR = Q3 - Q1\n    lower_bound = Q1 - 1.5 * IQR\n    upper_bound = Q3 + 1.5 * IQR\n    df = df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]'

In [65]:
# check for missing values
print(df.isnull().sum())

Auction                              0
VehYear                              0
Make                                 0
Color                                0
Transmission                         0
WheelType                            0
VehOdo                               0
Size                                 0
MMRAcquisitionAuctionAveragePrice    0
MMRAcquisitionAuctionCleanPrice      0
MMRAcquisitionRetailAveragePrice     0
MMRAcquisitionRetailCleanPrice       0
MMRCurrentAuctionAveragePrice        0
MMRCurrentAuctionCleanPrice          0
MMRCurrentRetailAveragePrice         0
MMRCurrentRetailCleanPrice           0
PRIMEUNIT                            0
AUCGUART                             0
VNST                                 0
VehBCost                             0
IsOnlineSale                         0
WarrantyCost                         0
IsBadBuy                             0
PurchaseMonth                        0
PurchaseYear                         0
dtype: int64


In [66]:
# one-hot encoding
df = pd.get_dummies(df)

In [67]:
# Are there any duplicate rows in the dataset?
duplicate_rows = df.duplicated()
print(f"Number of duplicate rows: {duplicate_rows.sum()}")

Number of duplicate rows: 41


In [20]:
# Save the cleaned dataset to a new CSV file
df.to_csv('cleaned_kick.csv', index=False)